#  Forward pass

__Автор задач: Блохин Н.В. (NVBlokhin@fa.ru)__

Материалы:
* Deep Learning with PyTorch (2020) Авторы: Eli Stevens, Luca Antiga, Thomas Viehmann
* https://pytorch.org/docs/stable/generated/torch.matmul.html
* https://machinelearningmastery.com/choose-an-activation-function-for-deep-learning/
* https://machinelearningmastery.com/loss-and-loss-functions-for-training-deep-learning-neural-networks/
* https://kidger.site/thoughts/jaxtyping/
* https://github.com/patrick-kidger/torchtyping/tree/master

## Задачи для совместного разбора

In [ ]:
pip install torchtyping

In [ ]:
from torchtyping import TensorType, patch_typeguard
from typeguard import typechecked
import torch as th

Scalar = TensorType[()]
patch_typeguard()

1\. Используя операции над матрицами и векторами из библиотеки `torch`, реализуйте нейрон с заданными весами `weights` и `bias`. Пропустите вектор `inputs` через нейрон и выведите результат.

In [ ]:
class Neuron:
    def __init__(self, n_features):
        # <создать атрибуты объекта weights и bias>
        self.weights = th.randn(n_features)
        self.bias = th.randn(1)
        print(self.bias, self.weights)
    def forward(self, inputs: TensorType["n_features"]) -> Scalar:
        return inputs.dot(self.weights) + self.bias


In [ ]:
import torch as th
inputs = th.tensor([1.0, 2.0, 3.0, 4.0])

In [ ]:
neuron = Neuron(4)
neuron.forward(inputs)

tensor([0.4084]) tensor([-0.1906, -0.6727,  0.4040,  1.0202])


tensor([4.1653])

2\. Используя операции над матрицами и векторами из библиотеки `torch`, реализуйте функцию активации ReLU:

![](https://wikimedia.org/api/rest_v1/media/math/render/svg/f4353f4e3e484130504049599d2e7b040793e1eb)

Создайте матрицу размера (4,3), заполненную числами из стандартного нормального распределения, и проверьте работоспособность функции активации.

In [ ]:
class ReLU:
    @typechecked
    def forward(self, inputs: TensorType) -> TensorType:
      return th.where(inputs > 0, inputs, th.zeros_like(inputs))

In [ ]:
inputs = th.randn(4,3)
inputs

tensor([[-0.8137,  0.9211,  1.2676],
        [-1.0951, -0.1611,  0.2793],
        [ 0.7792, -1.4202, -0.9903],
        [-0.4889,  0.3957,  0.5731]])

In [ ]:
relu = ReLU()
relu.forward(inputs)

tensor([[0.0000, 0.9211, 1.2676],
        [0.0000, 0.0000, 0.2793],
        [0.7792, 0.0000, 0.0000],
        [0.0000, 0.3957, 0.5731]])

3\. Используя операции над матрицами и векторами из библиотеки `torch`, реализуйте функцию потерь MSE:

![](https://wikimedia.org/api/rest_v1/media/math/render/svg/e258221518869aa1c6561bb75b99476c4734108e)
где $Y_i$ - правильный ответ для примера $i$, $\hat{Y_i}$ - предсказание модели для примера $i$, $n$ - количество примеров в батче.

In [ ]:
class MSELoss:
    @typechecked
    def forward(self, y_pred: TensorType["batch"], y_true: TensorType["batch"]) -> Scalar:
      #n = y_pred.size(0)
      return th.mean((y_true - y_pred)**2)

In [ ]:
y_pred = th.tensor([1.0, 3.0, 5.0])
y_true = th.tensor([2.0, 3.0, 4.0])

MSE = MSELoss()
MSE.forward(y_pred, y_true)

tensor(0.6667)

## Задачи для самостоятельного решения

### Cоздание полносвязных слоев

<p class="task" id="1"></p>

1\. Используя операции над матрицами и векторами из библиотеки `torch`, реализуйте полносвязный слой из `n_neurons` нейронов с `n_features` весами у каждого нейрона (инициализируются из стандартного нормального распределения) и опциональным вектором смещения.

$$y = xW^T + b$$

Пропустите вектор `inputs` через слой и выведите результат. Результатом прогона сквозь слой должна быть матрица размера `batch_size` x `n_neurons`.

- [ ] Проверено на семинаре

In [ ]:
class Linear:
    def __init__(self, n_neurons: int, n_features: int, bias: bool = False) -> None:
        self.weights = th.randn(n_features, n_neurons)
        self.bias = th.randn(n_neurons)

    def forward(self, inputs: TensorType["batch", "feats"]) -> TensorType["batch", "n_neurons"]:
        return th.mm(inputs,self.weights) + self.bias # ["bath", "n_features"] * (n_features, n_neurons)

In [ ]:
inputs = th.randn(5,4)
inputs

tensor([[ 0.7897, -0.1522,  0.1073, -0.7970],
        [ 0.5760, -1.2293, -0.3393, -0.2293],
        [ 0.6324,  0.3018, -0.5194, -0.7939],
        [ 0.1417, -0.5179,  0.4739, -0.0220],
        [ 0.9140, -1.3490,  0.7072,  1.4830]])

In [ ]:
linear = Linear(n_neurons=3, n_features=4, bias=True)
linear.forward(inputs)

tensor([[-0.2646, -0.5388, -1.6962],
        [ 0.9078, -0.6217, -1.0045],
        [-0.5356, -0.8042, -2.0240],
        [ 0.9446, -0.4265, -0.5751],
        [ 3.5197, -1.1376,  1.3323]])

<p class="task" id="2"></p>

2\. Используя решение предыдущей задачи, создайте 2 полносвязных слоя и пропустите тензор `inputs` последовательно через эти два слоя. Количество нейронов в первом слое выберите произвольно, количество нейронов во втором слое выберите так, чтобы результатом прогона являлась матрица `batch_size x 7`.

- [ ] Проверено на семинаре

In [ ]:
inputs = th.randn(5, 4)

l1 = Linear(6, 4, bias = True)
hidden = l1.forward(inputs)

l2 = Linear(7, 6, bias = True)
output = l2.forward(hidden)

print("Вход:\n", inputs)
print("Скрытый:\n", hidden)
print("Выход:\n", output)

Вход:
 tensor([[-0.0123,  0.3975, -1.7456,  1.1057],
        [-0.3267, -0.3923, -0.2043,  0.3871],
        [ 0.1550, -0.8367, -0.3016,  0.0687],
        [ 0.8248,  0.8480, -0.6528,  1.2626],
        [ 1.3592, -0.6689,  0.6032, -0.7273]])
Скрытый:
 tensor([[-1.3879, -0.8993,  1.8831, -0.4233,  1.1028, -1.7840],
        [ 0.9685,  0.8743, -1.6723,  0.9557, -1.2283, -1.5652],
        [-0.8494,  1.6407, -1.5607,  1.0171, -1.1592, -1.7334],
        [ 0.4851, -0.5470,  2.1367,  0.4721,  0.7164,  0.3332],
        [-0.2932,  3.8530, -1.0966,  2.0022, -1.5195, -0.4655]])
Выход:
 tensor([[-3.0600, -0.5007,  1.5177, -2.5148, -2.0492,  1.9220, -2.2374],
        [-3.6778,  4.7600,  3.1233,  3.6855, -2.7497, -2.6449,  6.5505],
        [-5.7316,  5.3376, -1.2969,  5.3072, -7.5936, -1.7329,  4.0418],
        [-0.2070, -0.5406,  0.7246, -0.9176,  1.4489, -0.4015, -0.9191],
        [-6.0647,  7.0082, -7.4369, 10.7367, -9.7676, -5.1169,  5.5780]])


### Создание функций активации

<p class="task" id="3"></p>

3\. Используя операции над матрицами и векторами из библиотеки `torch`, реализуйте функцию активации softmax:

![](https://wikimedia.org/api/rest_v1/media/math/render/svg/6d7500d980c313da83e4117da701bf7c8f1982f5)

$$\overrightarrow{x} = (x_1, ..., x_J)$$

Создайте матрицу размера (4,3), заполненную числами из стандартного нормального распределения, и проверьте работоспособность функции активации. Строки матрицы трактовать как выходы линейного слоя некоторого классификатора для 4 различных примеров. Функция должна применяться переданной на вход матрице построчно.

- [ ] Проверено на семинаре

In [ ]:
class Softmax:
    def forward(self, inputs: TensorType["batch", "feats"]) -> TensorType["batch", "feats"]:
      exp_inp = th.exp(inputs)
      sum_exp = th.sum(exp_inp, dim = 1, keepdim = True) # суммирует вдоль feats, сохраняет размерность
      return exp_inp/sum_exp

In [ ]:
input = th.randn(4, 3)
softmax = Softmax()
softmax.forward(input)

tensor([[0.2772, 0.3423, 0.3806],
        [0.6874, 0.1956, 0.1170],
        [0.4343, 0.5447, 0.0210],
        [0.8103, 0.0711, 0.1186]])

<p class="task" id="4"></p>

4 Используя операции над матрицами и векторами из библиотеки `torch`, реализуйте функцию активации ELU:

![](https://wikimedia.org/api/rest_v1/media/math/render/svg/eb23becd37c3602c4838e53f532163279192e4fd)

Создайте матрицу размера 4x3, заполненную числами из стандартного нормального распределения, и проверьте работоспособность функции активации.

- [ ] Проверено на семинаре

In [ ]:
class ELU:
    def __init__(self, alpha: float) -> None:
        self.alpha = alpha

    def forward(self, inputs: TensorType["batch", "feats"]) -> TensorType["batch", "feats"]:
        return th.where(inputs > 0, inputs, self.alpha * (th.exp(inputs) - 1))

In [ ]:
inputs = th.randn(4, 3)
inputs

tensor([[-1.7345, -1.2740,  2.2108],
        [ 0.3900, -0.6593, -1.0090],
        [-0.2433, -0.6857,  1.7816],
        [ 0.1517,  0.2935, -1.0729]])

In [ ]:
elu = ELU(alpha = 1.0)
elu.forward(inputs)

tensor([[-0.8235, -0.7203,  2.2108],
        [ 0.3900, -0.4828, -0.6354],
        [-0.2160, -0.4963,  1.7816],
        [ 0.1517,  0.2935, -0.6580]])

### Создание функции потерь

<p class="task" id="5"></p>

5 Используя операции над матрицами и векторами из библиотеки `torch`, реализуйте функцию потерь CrossEntropyLoss:

$$y_i = (y_{i,1},...,y_{i,k})$$

<img src="https://i.ibb.co/93gy1dN/Screenshot-9.png" width="200">

$$ CrossEntropyLoss = \frac{1}{n}\sum_{i=1}^{n}{L_i}$$
где $y_i$ - вектор правильных ответов для примера $i$, $\hat{y_i}$ - вектор предсказаний модели для примера $i$; $k$ - количество классов, $n$ - количество примеров в батче.

Создайте полносвязный слой с 2 нейронами и прогнать через него батч `inputs`. Полученный результат пропустите через функцию активации Softmax. Посчитайте значение функции потерь, трактуя вектор `y` как вектор правильных ответов.

- [ ] Проверено на семинаре

In [ ]:
class CrossEntropyLoss:
    def forward(self, y_pred: TensorType["batch", float], y_true: TensorType["batch", int]):
       return -th.mean(th.sum(y_true * th.log(y_pred), dim = 1))


In [ ]:
inputs = th.randn(3,4)
y_true = th.tensor([[1.0, 0.0],
                    [0.0, 1.0],
                    [0.5, 0.5]])
inputs

tensor([[ 0.5433,  0.7377, -1.5575, -0.5482],
        [-0.6487,  1.8326, -0.0351, -1.5609],
        [-0.0620, -0.5060, -2.9370, -0.5404]])

In [ ]:
linear = Linear(n_neurons = 2, n_features= 4, bias = True)
a = linear.forward(inputs)
a

tensor([[-0.5378,  2.2378],
        [-0.4151,  3.1209],
        [ 1.6768,  2.7456]])

In [ ]:
softmax = Softmax()
y_pred_1 = softmax.forward(a)
y_pred_1

tensor([[0.0587, 0.9413],
        [0.0283, 0.9717],
        [0.2556, 0.7444]])

In [ ]:
loss = CrossEntropyLoss()
loss.forward(y_pred_1, y_true)

tensor(1.2314)

<p class="task" id="6"></p>

6 Модифицируйте MSE, добавив L2-регуляризацию.

$$MSE_R = MSE + \lambda\sum_{i=1}^{m}w_i^2$$

где $\lambda$ - коэффициент регуляризации; $w_i$ - веса модели.

- [ ] Проверено на семинаре

In [ ]:
class MSERegularized:
    def __init__(self, lambda_: float) -> None:
        self.lambda_ = lambda_

    def data_loss(
            self,
            y_pred: TensorType["batch"],
            y_true: TensorType["batch"],
    ) -> Scalar:

        return th.mean((y_true - y_pred)**2)

    def reg_loss(self, weights: TensorType["batch", 1])  -> Scalar:

        return th.sum(weights ** 2)

    def forward(
        self,
        y_pred: TensorType["batch"],
        y_true: TensorType["batch"],
        weights: TensorType["batch", 1],
    ) -> Scalar:
        return self.data_loss(y_pred, y_true) + self.lambda_ * self.reg_loss(weights)

In [ ]:
y_pred = th.tensor([1.0, 3.0, 5.0])
y_true = th.tensor([2.0, 3.0, 4.0])
weights = th.tensor([0.1, 0.2, 0.3])
lambda_ = 0.01

In [ ]:
mse = MSERegularized(lambda_ = lambda_)
mse.forward(y_pred, y_true, weights)

tensor(0.6681)